# 🦐 SMARTAMBAK: OOD (Out-of-Distribution) Benchmark & Failure Analysis
Notebook ini didedikasikan untuk menguji ketahanan model YOLO terhadap gambar **Out-of-Distribution (OOD)** — yaitu gambar yang **100% bukan udang** (manusia, tangan, ikan, air, infrastruktur tambak, dll).

### Fitur Utama:
1. **Model & Dataset Flexibility:** Mudah mengganti path bobot model (`.pt`) dan folder dataset OOD yang ingin diuji.
2. **Comprehensive Summary Table:** Tabel rekapitulasi metrik *False Positive Rate (FPR %)* per kategori.
3. **Visualisasi Grid Gambar GAGAL (False Alarms):** Menampilkan foto-foto non-udang yang keliru terdeteksi sebagai udang beserta bounding box dan skor confidencenya untuk mempermudah analisis *root cause*.
4. **Export Laporan:** Menyimpan rekapitulasi evaluasi ke file CSV di folder `reports/`.

## Section 1: Inisialisasi Dependensi, Pemilihan Model & Dataset

In [1]:
import os, glob, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch
from ultralytics import YOLO

# =============================================================================
# 1. PILIH BOBOT MODEL (WEIGHTS PATH)
# =============================================================================
# Kosongkan string "" di bawah untuk otomatis menggunakan best.pt terbaru,
# atau isi path spesifik ke model yang ingin Anda uji.
MANUAL_WEIGHTS_PATH = "runs/detect/abiyamf/SMARTAMBAK/stage3-binary-null-20-05-38/weights/best.pt"

MODEL_PATH = None
if MANUAL_WEIGHTS_PATH and os.path.exists(MANUAL_WEIGHTS_PATH):
    MODEL_PATH = MANUAL_WEIGHTS_PATH
else:
    all_best_pts = glob.glob("runs/detect/abiyamf/SMARTAMBAK/*/weights/best.pt")
    if all_best_pts:
        all_best_pts.sort(key=os.path.getmtime, reverse=True)
        MODEL_PATH = all_best_pts[0]
    else:
        MODEL_PATH = "yolov8n.pt"

print(f"🎯 Model yang Dipilih  : {MODEL_PATH}")
model = YOLO(MODEL_PATH)

# =============================================================================
# 2. PILIH FOLDER DATASET OOD
# =============================================================================
# Opsi:
# - "dataset/OOD_TEST"     : 400 gambar OOD standar (manusia, tangan, ikan, air, dll)
# - "dataset/OOD_EXTENDED" : Gambar OOD baru (infrastruktur tambak, alat lab, burung, dll)
# - "ALL"                  : Menguji seluruh gambar dari kedua folder di atas
OOD_SOURCE = "ALL"

# Hyperparameter Uji Inferensi
CONF_THRESHOLD = 0.25   # Ambang batas deteksi (standar YOLO: 0.25)
IOU_THRESHOLD = 0.45    # NMS IoU threshold
IMGSZ = 640
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"📁 Sumber Dataset OOD  : {OOD_SOURCE}")
print(f"⚙️ Confidence Threshold: {CONF_THRESHOLD}")
print(f"💻 Device              : {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


🎯 Model yang Dipilih  : runs/detect/abiyamf/SMARTAMBAK/stage3-binary-null-20-05-38/weights/best.pt
📁 Sumber Dataset OOD  : ALL
⚙️ Confidence Threshold: 0.25
💻 Device              : 0 (NVIDIA GeForce RTX 5070 Ti)


## Section 2: Pengumpulan Daftar Gambar OOD & Eksekusi Inferensi Batch

In [2]:
# Mengumpulkan path gambar OOD beserta label kategorinya
image_records = []

def scan_ood_dir(base_dir):
    bpath = Path(base_dir)
    if not bpath.exists():
        return
    for p in bpath.glob("*/*.*"): # format: base_dir/kategori/gambar.jpg
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            image_records.append({
                "path": str(p),
                "category": p.parent.name,
                "source_folder": base_dir
            })

if OOD_SOURCE == "ALL":
    scan_ood_dir("dataset/OOD_TEST")
    scan_ood_dir("dataset/OOD_EXTENDED")
else:
    scan_ood_dir(OOD_SOURCE)

print(f"Total Gambar OOD Ditemukan: {len(image_records)}")
if image_records:
    df_images = pd.DataFrame(image_records)
    print("\nDistribusi Jumlah Gambar per Kategori:")
    display(df_images.groupby(["source_folder", "category"]).size().reset_index(name="jumlah_gambar"))
else:
    print("⚠️ Tidak ada gambar yang ditemukan. Pastikan folder dataset tersedia.")


Total Gambar OOD Ditemukan: 454

Distribusi Jumlah Gambar per Kategori:


,source_folder,category,jumlah_gambar
0,dataset/OOD_EXTENDED,alat_ukur_lab,2
1,dataset/OOD_EXTENDED,ikan_polikultur,30
2,dataset/OOD_EXTENDED,moluska_sefalopoda,22
3,dataset/OOD_TEST,crustacean,50
4,dataset/OOD_TEST,equipment,50
5,dataset/OOD_TEST,fish,50
6,dataset/OOD_TEST,hand,50
7,dataset/OOD_TEST,human,50
8,dataset/OOD_TEST,random_objects,50
9,dataset/OOD_TEST,rock_shell,50


In [3]:
# Menjalankan Batch Prediction secara hemat VRAM (Chunking)
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

results_records = []
failure_cases = []  # Menyimpan informasi gambar-gambar yang menghasilkan deteksi palsu

start_time = time.time()
CHUNK_SIZE = 32  # Memproses 32 gambar per batch untuk mencegah lonjakan memori GPU

print(f"🚀 Menjalankan inferensi pada {len(image_records)} gambar OOD (Chunk size: {CHUNK_SIZE})...")

for chunk_start in range(0, len(image_records), CHUNK_SIZE):
    chunk_records = image_records[chunk_start : chunk_start + CHUNK_SIZE]
    chunk_paths = [r["path"] for r in chunk_records]
    
    chunk_results = model.predict(
        source=chunk_paths,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMGSZ,
        device=DEVICE,
        verbose=False
    )
    
    for rec, res in zip(chunk_records, chunk_results):
        boxes = res.boxes
        num_boxes = len(boxes)
        has_fp = num_boxes > 0
        
        box_details = []
        max_conf = 0.0
        avg_conf = 0.0
        detected_classes = []
        
        if has_fp:
            confs = boxes.conf.cpu().numpy()
            cls_ids = boxes.cls.cpu().numpy().astype(int)
            xyxy_boxes = boxes.xyxy.cpu().numpy()
            
            max_conf = float(np.max(confs))
            avg_conf = float(np.mean(confs))
            detected_classes = [model.names[c] for c in cls_ids]
            
            for b_box, c_conf, c_name in zip(xyxy_boxes, confs, detected_classes):
                box_details.append({
                    "box": b_box,
                    "conf": float(c_conf),
                    "class": c_name
                })
                
            failure_cases.append({
                "path": rec["path"],
                "category": rec["category"],
                "source": rec["source_folder"],
                "num_detections": num_boxes,
                "max_conf": max_conf,
                "avg_conf": avg_conf,
                "detected_classes": detected_classes,
                "box_details": box_details
            })
            
        results_records.append({
            "path": rec["path"],
            "category": rec["category"],
            "source": rec["source_folder"],
            "has_fp": has_fp,
            "num_boxes": num_boxes,
            "max_conf": max_conf,
            "avg_conf": avg_conf,
            "detected_classes": detected_classes
        })
        
    # Bersihkan cache VRAM antar chunk
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

total_time = time.time() - start_time
fps = len(image_records) / total_time if total_time > 0 else 0
avg_ms = (total_time / len(image_records)) * 1000 if image_records else 0

print(f"✅ Inferensi Selesai dalam {total_time:.2f} detik ({avg_ms:.2f} ms/gambar, ~{fps:.1f} FPS)")

df_results = pd.DataFrame(results_records)
print(f"\nTotal Gambar False Alarm: {len(failure_cases)} / {len(image_records)}")


🚀 Menjalankan inferensi pada 454 gambar OOD...


[W920 14:59:16.996140058 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 3720347648 bytes (free: 1096220672, total: 16587751424).
[W920 14:59:17.210733618 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2975858688 bytes (free: 2184642560, total: 16587751424).
[W920 14:59:17.210832145 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2975858688 bytes (free: 2184642560, total: 16587751424).
[W920 14:59:17.228888269 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 5207228416 bytes (free: 2183987200, total: 16587751424).
[W920 14:59:17.228984907 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 5207228416 bytes (free: 2183987200, total: 16587751424).
[W920 14:59:17.481720327 CUDACachingAllocator.cpp:3934] memory allocation failed

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.77 GiB. GPU 0 has a total capacity of 15.45 GiB of which 2.04 GiB is free. Including non-PyTorch memory, this process has 12.46 GiB memory in use. Of the allocated memory 7.66 GiB is allocated by PyTorch, and 4.50 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## Section 3: Summary Table & Analisis False Positive Rate (FPR)

In [ ]:
# Menghitung metrik agregat per kategori OOD
summary_list = []
categories = sorted(df_results["category"].unique())

for cat in categories:
    sub = df_results[df_results["category"] == cat]
    total_imgs = len(sub)
    fp_imgs = sub["has_fp"].sum()
    total_fp_boxes = sub["num_boxes"].sum()
    fpr_pct = (fp_imgs / total_imgs) * 100 if total_imgs > 0 else 0.0
    
    fp_sub = sub[sub["has_fp"]]
    avg_conf = fp_sub["avg_conf"].mean() if len(fp_sub) > 0 else 0.0
    max_conf = fp_sub["max_conf"].max() if len(fp_sub) > 0 else 0.0
    
    all_classes = [c for sublist in fp_sub["detected_classes"] for c in sublist]
    most_common_class = max(set(all_classes), key=all_classes.count) if all_classes else "-"
    
    summary_list.append({
        "Kategori OOD": cat,
        "Total Gambar": total_imgs,
        "False Alarm (Img)": fp_imgs,
        "Total BBox Palsu": total_fp_boxes,
        "FPR (%)": round(fpr_pct, 2),
        "Rata-rata Conf": round(avg_conf, 3),
        "Max Conf": round(max_conf, 3),
        "Kelas Dominan Salah": most_common_class
    })

df_summary = pd.DataFrame(summary_list)

# Tambahkan baris total agregat
tot_imgs = len(df_results)
tot_fp = len(failure_cases)
tot_boxes = df_results["num_boxes"].sum()
overall_fpr = (tot_fp / tot_imgs) * 100 if tot_imgs > 0 else 0.0
all_fp_sub = df_results[df_results["has_fp"]]
ov_avg_conf = all_fp_sub["avg_conf"].mean() if len(all_fp_sub) > 0 else 0.0
ov_max_conf = all_fp_sub["max_conf"].max() if len(all_fp_sub) > 0 else 0.0

df_total_row = pd.DataFrame([{
    "Kategori OOD": "== KESELURUHAN (TOTAL) ==",
    "Total Gambar": tot_imgs,
    "False Alarm (Img)": tot_fp,
    "Total BBox Palsu": tot_boxes,
    "FPR (%)": round(overall_fpr, 2),
    "Rata-rata Conf": round(ov_avg_conf, 3),
    "Max Conf": round(ov_max_conf, 3),
    "Kelas Dominan Salah": "-"
}])

df_display_summary = pd.concat([df_summary, df_total_row], ignore_index=True)

print("=" * 80)
print(f"📊 TABEL RANGKUMAN UJI OOD BENCHMARK ({Path(MODEL_PATH).name})")
print("=" * 80)
display(df_display_summary.style.highlight_max(subset=["FPR (%)"], color="#fecaca"))


In [ ]:
# Grafik Bar Chart False Positive Rate (FPR %) per Kategori
plt.figure(figsize=(12, 6))
categories_plot = df_summary["Kategori OOD"]
fpr_plot = df_summary["FPR (%)"]

colors = ["#10b981" if val == 0.0 else "#ef4444" for val in fpr_plot]
bars = plt.barh(categories_plot, fpr_plot, color=colors, edgecolor="#334155")

plt.xlabel("False Positive Rate (%)", fontsize=12)
plt.title(f"OOD False Positive Rate (FPR %) per Kategori - Model: {Path(MODEL_PATH).name}", fontsize=14, fontweight="bold")
plt.axvline(x=overall_fpr, color="#3b82f6", linestyle="--", linewidth=2, label=f"Rata-rata Keseluruhan: {overall_fpr:.2f}%")
plt.grid(axis="x", linestyle="--", alpha=0.6)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.2, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va="center", fontweight="bold", fontsize=9)

plt.legend()
plt.tight_layout()
plt.show()


## Section 4: Visualisasi Grid Gambar yang GAGAL (False Alarms)
Bagian ini menampilkan seluruh foto non-udang yang keliru terdeteksi sebagai udang oleh model. Bounding box dan skor confidence digambar langsung pada foto untuk menganalisis mengapa model tertipu.

In [ ]:
if len(failure_cases) == 0:
    print("=" * 60)
    print("🎉 LUAR BIASA! TIDAK ADA GAMBAR YANG MENGHASILKAN FALSE ALARM.")
    print("Model berhasil menolak 100% objek Out-of-Distribution secara sempurna (FPR = 0.00%).")
    print("=" * 60)
else:
    max_display = min(len(failure_cases), 16)  # Batasi tampilan maksimal 16 gambar agar rapi
    cols = 4
    rows = int(np.ceil(max_display / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(18, 4.5 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = axes.reshape(rows, cols)
        
    print(f"⚠️ Menampilkan {max_display} dari {len(failure_cases)} kasus False Alarm:")
    
    for idx in range(rows * cols):
        r_idx = idx // cols
        c_idx = idx % cols
        ax = axes[r_idx, c_idx]
        
        if idx < max_display:
            item = failure_cases[idx]
            img = Image.open(item["path"]).convert("RGB")
            ax.imshow(img)
            
            # Gambar bounding box palsu
            for b_info in item["box_details"]:
                x1, y1, x2, y2 = b_info["box"]
                w = x2 - x1
                h = y2 - y1
                rect = patches.Rectangle((x1, y1), w, h, linewidth=2.5, edgecolor="#ef4444", facecolor="none")
                ax.add_patch(rect)
                
                # Label text
                label_text = f"{b_info['class']} {b_info['conf']*100:.1f}%"
                ax.text(
                    x1, max(0, y1 - 6),
                    label_text,
                    color="white",
                    fontsize=9,
                    fontweight="bold",
                    bbox=dict(facecolor="#ef4444", edgecolor="none", pad=2, alpha=0.9)
                )
                
            fname = Path(item["path"]).name
            ax.set_title(f"[{item['category']}]\n{fname}", fontsize=10, fontweight="bold", color="#b91c1c")
        else:
            ax.axis("off")
        ax.axis("off")
        
    plt.tight_layout()
    plt.show()


## Section 5: Distribusi Confidence & Ekspor Hasil Benchmark

In [ ]:
# 1. Plot Histogram Distribusi Confidence pada Deteksi Palsu
if len(failure_cases) > 0:
    all_fp_confs = [b["conf"] for fc in failure_cases for b in fc["box_details"]]
    plt.figure(figsize=(10, 5))
    plt.hist(all_fp_confs, bins=15, color="#f97316", edgecolor="#c2410c", alpha=0.8)
    plt.xlabel("Confidence Score Deteksi Palsu", fontsize=11)
    plt.ylabel("Jumlah Bounding Box Palsu", fontsize=11)
    plt.title("Distribusi Confidence Score False Alarm (Membantu Menentukan Ambang Threshold Operasional)", fontsize=12, fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

# 2. Simpan Rekapitulasi ke CSV
os.makedirs("reports", exist_ok=True)
report_path = "reports/ood_benchmark_summary.csv"
df_display_summary.to_csv(report_path, index=False)
print(f"💾 Laporan OOD berhasil diekspor ke: {report_path}")
